# 03 · Panel 2020 → 2025 y matriz de éxito

**Objetivo:** saber qué le pasó a cada restaurante entre las dos ediciones con Censo Económico (04/2020 y 05/2025) y resumirlo en una matriz **hexágono × tipo de restaurante**, que es lo que aprenderá el modelo de recomendación.

**Por qué no basta comparar `id`:** un `id` que desaparece puede ser un cierre, el mismo negocio con clave nueva o un local que ahora ocupa otro negocio. Por eso a cada restaurante que desaparece se le busca un **sucesor** en el mismo lugar.

| Categoría | Regla |
|---|---|
| **Continúa** | Mismo `id` en 2025, sigue siendo restaurante |
| **Continúa (id nuevo)** | Sin `id`, pero en 2025 hay un negocio nuevo en el mismo lugar con nombre casi igual |
| **Cambió de giro** | Mismo `id`, pero en 2025 ya no es restaurante |
| **Reemplazo, mismo tipo** | Salió, y en su lugar hay un restaurante nuevo del mismo tipo |
| **Reemplazo, otro restaurante** | Salió, y en su lugar hay un restaurante de otro tipo |
| **Reemplazo, otro giro** | Salió, y en su lugar hay un negocio que no es restaurante |
| **Salida sin sucesor** | Salió y no hay ningún negocio nuevo en ese lugar |

"Mismo lugar" quiere decir **misma dirección** (calle y número exterior, y número interior si ambos lo tienen) o **a 10 m o menos**.

## 0. Librerías y rutas

In [ ]:
# Requiere: conda install -n tesis-qro -c conda-forge rapidfuzz
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h3
from rapidfuzz import fuzz
import mapas

ROOT = Path.cwd().parent
PROC, FIG = ROOT / "data/processed", ROOT / "figures"
VAL = ROOT / "data/validacion"
VAL.mkdir(parents=True, exist_ok=True)

## 1. Configuración

Los umbrales son decisiones metodológicas: conviene probar su sensibilidad (sección 8).

In [ ]:
BASE, FIN = "2020-04", "2025-05"
RES = 8                  # hexágono de la matriz (~0.74 km²)
RADIO_M = 10             # distancia máxima para considerar "mismo lugar"
UMBRAL_NOMBRE = 80       # similitud de nombre (0–100) para "mismo negocio, id nuevo"

TIPOS_RESTAURANTE = {
    "722511": "A la carta / comida corrida", "722512": "Mariscos", "722513": "Antojitos",
    "722514": "Tacos y tortas", "722515": "Cafeterías y neverías", "722516": "Autoservicio",
    "722517": "Pizzas, hamburguesas y pollos", "722518": "Otros para llevar",
    "722519": "Otros de consumo inmediato", "722411": "Bares y centros nocturnos",
    "722412": "Bares y centros nocturnos",
}

## 2. Cargar las dos ediciones

- `base`: restaurantes de 2020 (a los que se les sigue la pista).
- `fin`: **todo** el DENUE de 2025, porque el sucesor de un restaurante puede ser cualquier negocio.

In [ ]:
cols = ["id", "nom_estab", "raz_social", "codigo_act", "nom_vial", "numero_ext", "numero_int",
        "latitud", "longitud", "fecha_alta", "mun_nombre"]

def cargar(edicion):
    df = pd.read_parquet(PROC / f"denue_zmq_{edicion}.parquet")
    return df[[c for c in cols if c in df.columns]].reset_index(drop=True)

base_todo, fin = cargar(BASE), cargar(FIN)
base = base_todo[base_todo["codigo_act"].isin(TIPOS_RESTAURANTE)].copy()
base["tipo"] = base["codigo_act"].map(TIPOS_RESTAURANTE)
fin["tipo"] = fin["codigo_act"].map(TIPOS_RESTAURANTE)          # NaN si no es restaurante
fin["es_rest"] = fin["tipo"].notna()
print(f"Restaurantes {BASE}: {len(base):,} · negocios {FIN}: {len(fin):,} (restaurantes: {fin['es_rest'].sum():,})")

## 3. Funciones de enlace

- `direccion`: clave normalizada calle + número exterior (+ interior).
- `nombre_norm`: quita acentos, signos y palabras vacías ("S.A. de C.V.", artículos).
- `similitud`: el máximo entre nombre comercial y razón social. Hay dos opciones: `token_set_ratio` tolera palabras en distinto orden ("TACOS EL GÜERO" vs "EL GÜERO TACOS"); `ratio` es más estricta. Se usa la primera.

In [ ]:
import unicodedata

def _limpio(s):
    s = unicodedata.normalize("NFKD", str(s or "")).encode("ascii", "ignore").decode().upper()
    return re.sub(r"[^A-Z0-9 ]", " ", s)

VACIAS = r"\b(SA|DE|CV|S|A|C|V|RL|SC|SAPI|SAS|LA|EL|LOS|LAS|Y)\b"

def nombre_norm(s):
    return " ".join(re.sub(VACIAS, " ", _limpio(s)).split())

def direccion(calle, ext, inte):
    calle = " ".join(_limpio(calle).split())
    ext = re.sub(r"\D", "", str(ext or ""))
    inte = re.sub(r"[^A-Z0-9]", "", _limpio(inte))
    if not calle or not ext or ext == "0":
        return None, None
    return f"{calle}#{ext}", (inte or None)

def similitud(a_nom, a_raz, b_nom, b_raz):
    s = fuzz.token_set_ratio(a_nom, b_nom)
    if a_raz and b_raz:
        s = max(s, fuzz.token_set_ratio(a_raz, b_raz))
    return s

def distancia_m(la1, lo1, la2, lo2):
    return 6_371_000 * np.hypot(np.radians(la2 - la1), np.radians(lo2 - lo1) * np.cos(np.radians(la1)))

for df in (base, fin):
    df["nom_n"] = df["nom_estab"].map(nombre_norm)
    df["raz_n"] = df["raz_social"].map(nombre_norm) if "raz_social" in df else ""
    dirs = [direccion(c, e, i) for c, e, i in zip(df["nom_vial"], df["numero_ext"], df.get("numero_int", [None] * len(df)))]
    df["dir"] = [d for d, _ in dirs]
    df["dir_int"] = [i for _, i in dirs]
    df["h12"] = [h3.latlng_to_cell(la, lo, 12) for la, lo in zip(df["latitud"], df["longitud"])]

## 4. Clasificar a cada restaurante de 2020

1. Los que tienen el mismo `id` en 2025: **continúa** o **cambió de giro**.
2. A los que desaparecen se les buscan candidatos entre los negocios **nuevos** de 2025 (ids que no existían en 2020), en la misma dirección o a ≤ 10 m. Si ambos tienen número interior y es distinto, son locales diferentes de la misma plaza y no cuentan.
3. Con candidatos: si el nombre es casi igual, es el mismo negocio con `id` nuevo; si no, es un reemplazo.

Un negocio nuevo solo puede ser "el mismo negocio con id nuevo" de un restaurante (enlace uno a uno). Como reemplazo, en cambio, puede contar para varios: en una plaza un local nuevo puede ocupar el espacio de dos.

In [ ]:
ids_fin = set(fin["id"])
fin_por_id = fin.set_index("id")
nuevos = fin[~fin["id"].isin(set(base_todo["id"]))]
idx_dir = nuevos.dropna(subset=["dir"]).groupby("dir").indices
idx_h12 = nuevos.groupby("h12").indices
N = nuevos.reset_index(drop=True)

def candidatos(r):
    cand = set()
    if r.dir is not None:
        cand |= set(idx_dir.get(r.dir, []))
    for c in h3.grid_disk(r.h12, 1):
        cand |= set(idx_h12.get(c, []))
    fuera = []
    for i in cand:
        n = N.iloc[i]
        mismo_dir = r.dir is not None and n["dir"] == r.dir
        if not mismo_dir and distancia_m(r.latitud, r.longitud, n["latitud"], n["longitud"]) > RADIO_M:
            continue
        if r.dir_int and n["dir_int"] and r.dir_int != n["dir_int"]:
            continue                                 # otro local de la misma plaza
        fuera.append(i)
    return fuera

estado, sucesor, sim_max = [], [], []
usados_reid = set()
for r in base.itertuples():
    if r.id in ids_fin:
        estado.append("Continúa" if fin_por_id.at[r.id, "es_rest"] else "Cambió de giro")
        sucesor.append(r.id); sim_max.append(np.nan)
        continue
    cand = candidatos(r)
    if not cand:
        estado.append("Salida sin sucesor"); sucesor.append(None); sim_max.append(np.nan)
        continue
    sims = {i: similitud(r.nom_n, r.raz_n, N.at[i, "nom_n"], N.at[i, "raz_n"]) for i in cand}
    mejor = max(sims, key=sims.get)
    if sims[mejor] >= UMBRAL_NOMBRE and mejor not in usados_reid:
        usados_reid.add(mejor)
        estado.append("Continúa (id nuevo)")
    elif any(N.at[i, "tipo"] == r.tipo for i in cand):
        estado.append("Reemplazo, mismo tipo")
    elif any(N.at[i, "es_rest"] for i in cand):
        estado.append("Reemplazo, otro restaurante")
    else:
        estado.append("Reemplazo, otro giro")
    sucesor.append(N.at[mejor, "id"]); sim_max.append(sims[mejor])

base["estado"], base["sucesor_id"], base["similitud"] = estado, sucesor, sim_max
ids_reid = set(N.loc[list(usados_reid), "id"])

ORDEN = ["Continúa", "Continúa (id nuevo)", "Cambió de giro", "Reemplazo, mismo tipo",
         "Reemplazo, otro restaurante", "Reemplazo, otro giro", "Salida sin sucesor"]
tab = base["estado"].value_counts().reindex(ORDEN, fill_value=0).to_frame("restaurantes")
tab["%"] = (tab["restaurantes"] / tab["restaurantes"].sum() * 100).round(1)
tab

## 5. Entradas: restaurantes de 2025 que no estaban en 2020

Se excluyen los que en realidad son un restaurante de 2020 con `id` nuevo. Cada entrada se marca según si ocupa un local que dejó un restaurante (**en local de restaurante**) o es una ubicación nueva.

Cautela: el Censo Económico 2024 también incorpora negocios que ya existían pero no estaban en el directorio. `fecha_alta` no lo resuelve, porque es la fecha en que entró al DENUE, no la de apertura.

In [ ]:
ent = fin[fin["es_rest"] & ~fin["id"].isin(set(base_todo["id"])) & ~fin["id"].isin(ids_reid)].copy()
suc_reemplazo = set(base.loc[base["estado"].str.startswith("Reemplazo"), "sucesor_id"].dropna())
ent["en_local_de_restaurante"] = ent["id"].isin(suc_reemplazo)
print(f"Entradas de restaurantes {BASE} → {FIN}: {len(ent):,}")
print(f"  en un local que dejó un restaurante: {ent['en_local_de_restaurante'].sum():,} "
      f"({ent['en_local_de_restaurante'].mean():.1%})")
ent.groupby("tipo").size().sort_values(ascending=False).rename("entradas")

## 6. La matriz hexágono × tipo

Por cada hexágono y tipo de restaurante:

| Columna | Qué es |
|---|---|
| `n_base` | Restaurantes de ese tipo en 2020 |
| `continuan` | De ellos, cuántos siguen en 2025 (incluye id nuevo) |
| `permanencia` | `continuan / n_base` (vacío si `n_base = 0`) |
| `reemplazo_mismo_tipo` | Salieron y su local lo tomó otro del mismo tipo |
| `salidas` | Todos los que ya no están (incluye cambio de giro y reemplazos) |
| `entradas` | Restaurantes nuevos de ese tipo en 2025 |
| `neto` | `entradas − salidas` |

La forma final del "éxito" que aprenderá el modelo se define en el notebook 04. Aquí se guardan los componentes para poder probar varias definiciones.

In [ ]:
def a_hex(df, res=RES):
    return [h3.latlng_to_cell(la, lo, res) for la, lo in zip(df["latitud"], df["longitud"])]

base["h3"], ent["h3"] = a_hex(base), a_hex(ent)
base["continua"] = base["estado"].isin(["Continúa", "Continúa (id nuevo)"])
base["sale"] = ~base["continua"]
base["reemp_mismo"] = base["estado"] == "Reemplazo, mismo tipo"

m_base = base.groupby(["h3", "tipo"]).agg(n_base=("id", "size"), continuan=("continua", "sum"),
                                          salidas=("sale", "sum"), reemplazo_mismo_tipo=("reemp_mismo", "sum"))
m_ent = ent.groupby(["h3", "tipo"]).agg(entradas=("id", "size"),
                                        entradas_en_local_rest=("en_local_de_restaurante", "sum"))

# Todas las combinaciones hexágono × tipo de los hexágonos de la tabla base 2020 (incluye ceros)
hex_base = pd.read_parquet(PROC / f"hex_res{RES}_base_2020.parquet")["h3"]
tipos = sorted(set(TIPOS_RESTAURANTE.values()))
celdas = pd.Index(sorted(set(hex_base) | set(m_base.index.get_level_values(0)) | set(m_ent.index.get_level_values(0))), name="h3")
matriz = (pd.DataFrame(index=pd.MultiIndex.from_product([celdas, tipos], names=["h3", "tipo"]))
            .join(m_base).join(m_ent).fillna(0).astype(int))
matriz["permanencia"] = np.where(matriz["n_base"] > 0, matriz["continuan"] / matriz["n_base"].replace(0, 1), np.nan)
matriz["neto"] = matriz["entradas"] - matriz["salidas"]

print(f"Matriz: {len(celdas):,} hexágonos × {len(tipos)} tipos = {len(matriz):,} celdas")
print(f"Celdas con al menos un restaurante en 2020: {(matriz['n_base'] > 0).mean():.1%}")
print(f"Celdas con al menos una entrada: {(matriz['entradas'] > 0).mean():.1%}")
matriz.sum(numeric_only=True).drop("permanencia").astype(int)

## 7. Primeras lecturas

### 7a. Qué pasó con los restaurantes de 2020, por tipo

In [ ]:
COLORES_ESTADO = {"Continúa": "#14213D", "Continúa (id nuevo)": "#2E6F95", "Cambió de giro": "#8DB7D3",
                  "Reemplazo, mismo tipo": "#E8A25F", "Reemplazo, otro restaurante": "#F3C89A",
                  "Reemplazo, otro giro": "#C9C2B4", "Salida sin sucesor": "#A8551A"}
por_tipo = (base.groupby(["tipo", "estado"]).size().unstack(fill_value=0)
                .reindex(columns=ORDEN, fill_value=0))
por_tipo = por_tipo.div(por_tipo.sum(axis=1), axis=0).sort_values("Continúa")

fig, ax = plt.subplots(figsize=(12, 7))
por_tipo.plot(kind="barh", stacked=True, ax=ax, width=0.7, color=[COLORES_ESTADO[c] for c in ORDEN])
ax.set_xlim(0, 1); ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_xlabel(f"% de los restaurantes de {BASE}"); ax.set_ylabel("")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False, title=f"Situación en {FIN}")
ax.set_title(f"¿Qué pasó con los restaurantes de {BASE}?", loc="left")
plt.tight_layout(); plt.savefig(FIG / "06_transiciones_por_tipo.png", dpi=200, bbox_inches="tight"); plt.show()
(por_tipo * 100).round(1)

### 7b. Adelanto de la hipótesis 2: ¿entrar es lo mismo que quedarse?

Por tipo, correlación de rangos (Spearman, calculada como Pearson sobre rangos para no depender de scipy) entre hexágonos de la tasa de entrada (entradas por restaurante existente) y la permanencia. Si es baja, las zonas que atraen restaurantes no son las mismas donde sobreviven.

In [ ]:
h2 = matriz[matriz["n_base"] >= 3].copy()            # al menos 3 para que la permanencia signifique algo
h2["tasa_entrada"] = h2["entradas"] / h2["n_base"]
corr = (h2.groupby("tipo")
          .apply(lambda d: pd.Series({"hexagonos": len(d),
                                      "spearman": d["tasa_entrada"].rank().corr(d["permanencia"].rank())}))
          .sort_values("spearman"))
corr.round(2)

### 7c. Mapa de entradas de restaurantes

In [ ]:
e_hex = matriz.groupby("h3")["entradas"].sum()
e_hex = e_hex[e_hex > 0]
BORDES = [1, 2, 4, 8, 16, 32, e_hex.max() + 1]
ETIQ = ["1", "2–3", "4–7", "8–15", "16–31", f"32–{e_hex.max()}"]
cmap, norm = mapas.escala_por_clases(BORDES, ["#EAF1F6", "#BFD6E6", "#8DB7D3", "#4F8DB8", "#2E6F95", "#14213D"])
fig, ax, polys = mapas.base_mapa(e_hex.index)
coll = mapas.capa_hex(ax, polys, array=e_hex.values, cmap=cmap, norm=norm)
mapas.referencias(ax)
mapas.barra_clases(fig, ax, coll, BORDES, ETIQ, "Restaurantes nuevos por hexágono")
ax.set_title(f"Entradas de restaurantes · {BASE} → {FIN}", loc="left", fontsize=13)
plt.tight_layout(); plt.savefig(FIG / "07_mapa_entradas.png", dpi=200, bbox_inches="tight"); plt.show()

## 8. Sensibilidad y validación manual

**Sensibilidad:** ¿cuánto cambian las categorías si el radio es 5, 10 o 20 m? Si cambian mucho, el resultado depende demasiado de una decisión arbitraria.

**Validación:** se exporta una muestra aleatoria de 25 casos por categoría a `data/validacion/muestra_transiciones.csv`. Ábrela en Excel, llena la columna `correcto` (1 o 0) revisando nombres y direcciones (y Google Maps o Street View si hace falta), y con eso se estima qué tan seguido acierta cada categoría. Ese porcentaje va en la tesis.

In [ ]:
def recontar(radio):
    global RADIO_M
    original, RADIO_M = RADIO_M, radio
    salidas = base[~base["id"].isin(ids_fin)]
    res = []
    for r in salidas.itertuples():
        cand = candidatos(r)
        res.append("Salida sin sucesor" if not cand else "Con sucesor en el lugar")
    RADIO_M = original
    return pd.Series(res).value_counts(normalize=True).mul(100).round(1)

pd.DataFrame({f"{r} m": recontar(r) for r in (5, 10, 20)})

In [ ]:
muestra = (base[base["estado"] != "Continúa"]
             .groupby("estado", group_keys=False)
             .apply(lambda d: d.sample(min(len(d), 25), random_state=42)))
suc = fin.set_index("id")[["nom_estab", "codigo_act", "nom_vial", "numero_ext", "numero_int"]].add_suffix("_2025")
muestra = (muestra[["id", "estado", "similitud", "nom_estab", "tipo", "nom_vial", "numero_ext", "numero_int",
                    "latitud", "longitud", "sucesor_id"]]
             .merge(suc, left_on="sucesor_id", right_index=True, how="left"))
muestra["google_maps"] = "https://www.google.com/maps?q=" + muestra["latitud"].astype(str) + "," + muestra["longitud"].astype(str)
muestra["correcto"] = ""
muestra.to_csv(VAL / "muestra_transiciones.csv", index=False, encoding="utf-8-sig")
print(f"Muestra para revisar: {len(muestra)} casos → {VAL / 'muestra_transiciones.csv'}")

## 9. Guardar

In [ ]:
base.drop(columns=["nom_n", "raz_n", "h12"]).to_parquet(PROC / "transiciones_restaurantes_2020_2025.parquet", index=False)
ent.drop(columns=["nom_n", "raz_n", "h12"]).to_parquet(PROC / "entradas_restaurantes_2020_2025.parquet", index=False)
matriz.reset_index().to_parquet(PROC / f"matriz_hex{RES}_tipo_2020_2025.parquet", index=False)
print("Guardados: transiciones, entradas y matriz hexágono × tipo")